# Pipeline: Google Sheet (DSNS + Cấp đơn) → Supabase (Postgres) → 3 Query thi đua → Excel

Bản chạy trên **Google Colab**, dữ liệu nguồn lấy trực tiếp từ **2 Google Sheet**, nạp vào **Supabase** (Postgres thật, nên 3 câu query `.sql` gốc chạy nguyên không cần sửa cú pháp).

Các bước (bật/tắt bằng cờ ở ô CẤU HÌNH):

1. **Đọc + làm sạch DSNS** từ Google Sheet → ghi đè bảng `ds_nhan_su_affina` trên Supabase.
2. **Đọc + làm sạch Cấp đơn** (7 sheet: Sức khỏe, Xe máy, BHYT/BHXH, Ô tô, Du lịch, TNSP, Rủi ro) → gộp → ghi đè bảng `union_all_data_cap_don` trên Supabase.
3. **Chạy 3 câu query** (`Contest_GDKD`, `Contest_CVKD_latest`, `Contest_TPKD_latest`) — đã nhúng sẵn nội dung trong notebook — xuất **1 file Excel gồm 3 sheet**.

> ⚠️ Trước khi chạy: điền connection string Supabase ở ô CẤU HÌNH bên dưới (Project Settings → Database → Connection string, chọn **Session pooler** hoặc **Direct connection**).
> ⚠️ Đảm bảo tài khoản Google bạn dùng để authenticate trong Colab có quyền xem 2 Google Sheet này (nếu sheet ở chế độ riêng tư).

In [ ]:
# ============================================================
# CẤU HÌNH CHUNG — chỉnh ở đây, không cần sửa các ô bên dưới
# ============================================================

# --- Cờ bật/tắt từng bước ---
RUN_UPDATE_DSNS   = True   # Bước 1: cập nhật bảng ds_nhan_su_affina
RUN_UPDATE_CAPDON = True   # Bước 2: cập nhật bảng union_all_data_cap_don
RUN_QUERIES       = True   # Bước 3: chạy 3 query + xuất Excel

# --- Google Sheet nguồn ---
DSNS_SHEET_URL   = "https://docs.google.com/spreadsheets/d/15LNUJMAU7adQfJTQHEKovGybQ_jRYd7l/edit?gid=322065732#gid=322065732"
DSNS_SHEET_NAME  = "DSNS AGENCY 2025"   # tên sheet (tab) chứa danh sách nhân sự

CAPDON_SHEET_URL = "https://docs.google.com/spreadsheets/d/1qc_QhrvpoLLp6w9RkGBEkm8qBO49GJE8oMlwkCdJOsk/edit?gid=1594679300#gid=1594679300"
# File cấp đơn cần có đủ các sheet (tab) sau, đúng tên:
#   'Sức khỏe', 'Thông tin cấp Bảo hiểm xe máy', 'BHYTBHXH',
#   'Bao hiem oto', 'Du lịch', 'Trách nhiệm sản phẩm', 'Bảo hiểm rủi ro'

# --- Thông tin kết nối Supabase (Postgres) ---
# Lấy tại: Supabase Project -> Project Settings -> Database -> Connection string
SUPABASE_USER     = "postgres"                    # hoặc postgres.<project-ref> nếu dùng pooler
SUPABASE_PASSWORD = "<mật khẩu database supabase>"
SUPABASE_HOST     = "<project-ref>.supabase.co"   # hoặc host của pooler, vd: aws-0-ap-southeast-1.pooler.supabase.com
SUPABASE_PORT     = "5432"                        # 5432 = direct connection, 6543 = transaction pooler
SUPABASE_DB       = "postgres"

# --- Link 3 file .sql (Google Drive public hoặc GitHub raw) ---
# Hỗ trợ:
#   - Link Google Drive share (dạng .../file/d/<ID>/view?usp=sharing) -> tự chuyển sang link tải trực tiếp
#   - Link GitHub thường (.../blob/...) -> tự chuyển sang raw.githubusercontent.com
#   - Link raw/trực tiếp khác (raw.githubusercontent.com, link tải .txt/.sql công khai...) -> dùng nguyên
SQL_GDKD_URL        = "<link Contest_GDKD.sql>"
SQL_CVKD_LATEST_URL = "<link Contest_CVKD_latest.sql>"
SQL_TPKD_LATEST_URL = "<link Contest_TPKD_latest.sql>"

# --- File Excel kết quả cuối cùng ---
OUTPUT_EXCEL_PATH = "Ket_qua_thi_dua.xlsx"


In [ ]:
# Cài thư viện cần thiết
%pip install pandas sqlalchemy psycopg2-binary openpyxl gspread --quiet


In [ ]:
import io
import re
import ast
import operator
import warnings
import unicodedata
from decimal import Decimal

import pandas as pd
from sqlalchemy import create_engine, text

# --- Xác thực Google (để tải Google Sheet, kể cả sheet riêng tư) ---
from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession

auth.authenticate_user()
_creds, _ = google.auth.default()
_gsession = AuthorizedSession(_creds)


def _extract_sheet_id(url: str) -> str:
    m = re.search(r"/d/([a-zA-Z0-9-_]+)", url)
    if not m:
        raise ValueError(f"Không tìm thấy Sheet ID trong URL: {url}")
    return m.group(1)


def download_google_sheet_as_excel_bytes(sheet_url: str) -> bytes:
    """Tải toàn bộ Google Sheet (mọi tab) dưới dạng file .xlsx (bytes)."""
    sheet_id = _extract_sheet_id(sheet_url)
    export_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    resp = _gsession.get(export_url)
    resp.raise_for_status()
    return resp.content


# --- Kết nối Supabase (Postgres) ---
engine = create_engine(
    f"postgresql+psycopg2://{SUPABASE_USER}:{SUPABASE_PASSWORD}"
    f"@{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}?sslmode=require"
)
with engine.connect() as _conn:
    _conn.execute(text("SELECT 1"))
print("✔ Đã kết nối thành công tới Supabase:", SUPABASE_HOST)


## Bước 1 — Cập nhật DSNS (`ds_nhan_su_affina`)

In [ ]:
def convert_columns_channel(row_values_channal, row_values_quan_ly):
    if pd.isna(row_values_channal) or row_values_channal == '':
        return "Core Agency"

    channal = str(row_values_channal).strip()
    channal_lower = channal.lower()

    if channal_lower in ['core agency', 'standard']:
        return "Core Agency"

    if 'tsa' in channal_lower:
        return "CTV_TSA (TSA 2)" if '2' in channal else "TSA"

    if channal == "Elite":
        return "Elite"

    if channal == "H.O":
        return "H.O"

    if channal == 'Neo':
        return "Neo"

    if channal == 'TSA':
        return "TSA"

    return "Core Agency"


def convert_date(date_str):
    try:
        date_str = str(date_str).strip()

        if '/' in date_str:
            parts = date_str.split('/')
            if len(parts) == 3:
                day, month, year = parts
                return f"{year.zfill(4)}-{month.zfill(2)}-{day.zfill(2)}"

        return pd.to_datetime(date_str, dayfirst=True, errors='coerce')

    except Exception as e:
        print(f"Lỗi chuyển đổi ngày: {date_str} -> {e}")
        return pd.NaT


def fix_inconsistent_dates(date_str):
    try:
        if isinstance(date_str, pd.Timestamp):
            return date_str if date_str.year > 1900 else pd.NaT

        date_str = str(date_str).strip()

        try:
            parsed = pd.to_datetime(date_str, format="%Y-%m-%d %H:%M:%S", errors="raise")
            return parsed
        except Exception:
            pass

        if '/' in date_str:
            parts = date_str.split('/')
            if len(parts) == 3:
                d, m, y = parts
                if len(y) == 2:
                    y = '20' + y
                return pd.to_datetime(f"{y}-{m.zfill(2)}-{d.zfill(2)}")

        return pd.to_datetime(date_str, dayfirst=True, errors='coerce')

    except Exception as e:
        print(f"Error parsing date '{date_str}': {e}")
        return pd.NaT


def clean_whitespace_and_newlines_dsns(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.replace(r'[\n\r]+', ' ', regex=True).str.strip()
    return df


In [ ]:
if RUN_UPDATE_DSNS:
    dsns_bytes = download_google_sheet_as_excel_bytes(DSNS_SHEET_URL)

    df_nhan_su_affina_test = pd.read_excel(
        io.BytesIO(dsns_bytes), dtype=str, sheet_name=DSNS_SHEET_NAME
    )
    df_nhan_su_affina_test = df_nhan_su_affina_test.dropna(subset='Họ tên')
    df_nhan_su_affina_test['Họ tên'] = df_nhan_su_affina_test['Họ tên'].str.strip()
    df_nhan_su_affina_test['QUẢN LÝ CẤP 1 (BDM)'] = (
        df_nhan_su_affina_test['QUẢN LÝ CẤP 1 (BDM)']
        .str.replace(r'[\n\r]+', ' ', regex=True).str.strip()
    )
    df_nhan_su_affina_test['QUẢN LÝ CẤP 2 (BDD)'] = (
        df_nhan_su_affina_test['QUẢN LÝ CẤP 2 (BDD)']
        .str.replace(r'[\n\r]+', ' ', regex=True).str.strip()
    )
    df_nhan_su_affina_test['Quản lý Cấp 3 (BDH)'] = (
        df_nhan_su_affina_test['Quản lý Cấp 3 (BDH)']
        .str.replace(r'[\n\r]+', ' ', regex=True).str.strip()
    )

    df_nhan_su_affina_test = clean_whitespace_and_newlines_dsns(df_nhan_su_affina_test)

    df_nhan_su_affina_test['Channel'] = df_nhan_su_affina_test.apply(
        lambda row: convert_columns_channel(row['Channal'], row['Quản lý Cấp 3 (BDH)']),
        axis=1,
    )

    df_nhan_su_affina_test['Thời gian bắt đầu'] = df_nhan_su_affina_test.apply(
        lambda row: fix_inconsistent_dates(row['Thời gian bắt đầu']), axis=1
    )
    df_nhan_su_affina_test['Ngày hiệu lực chức danh'] = df_nhan_su_affina_test[
        'Ngày hiệu lực chức danh'
    ].apply(fix_inconsistent_dates)
    df_nhan_su_affina_test['Ngày Sinh'] = df_nhan_su_affina_test.apply(
        lambda row: convert_date(row['Ngày Sinh']), axis=1
    )
    df_nhan_su_affina_test['Ngày Sinh'] = df_nhan_su_affina_test.apply(
        lambda row: fix_inconsistent_dates(row['Ngày Sinh']), axis=1
    )
    df_nhan_su_affina_test['Điện thoại'] = (
        df_nhan_su_affina_test['Điện thoại'].astype(str).str.lstrip('0')
    )
    df_nhan_su_affina_test['Code UM'] = (
        df_nhan_su_affina_test['Code UM'].astype(str).str.lstrip('0')
    )
    df_nhan_su_affina_test['Người giới thiệu'] = (
        df_nhan_su_affina_test['Người giới thiệu'].astype(str).str.lstrip('0')
    )
    df_nhan_su_affina_test['Thời gian kết thúc'] = df_nhan_su_affina_test.apply(
        lambda row: convert_date(row['Thời gian kết thúc']), axis=1
    )
    df_nhan_su_affina_test['Thời gian kết thúc'] = df_nhan_su_affina_test.apply(
        lambda row: fix_inconsistent_dates(row['Thời gian kết thúc']), axis=1
    )

    df_nhan_su_affina_test.columns = df_nhan_su_affina_test.columns.str.strip()

    df_nhan_su_affina_test.to_sql(
        'ds_nhan_su_affina', index=False, if_exists='replace', con=engine
    )

    print(f"✔ Đã cập nhật {len(df_nhan_su_affina_test)} dòng vào bảng ds_nhan_su_affina (Supabase)")
else:
    print("⏭ Bỏ qua Bước 1 (RUN_UPDATE_DSNS = False)")


## Bước 2 — Cập nhật Cấp đơn (`union_all_data_cap_don`)

In [ ]:
def clean_contract(value):
    if pd.isna(value):
        return value
    text_ = str(value)
    text_ = unicodedata.normalize('NFC', text_)
    text_ = re.sub(r'[–—−-]', '-', text_)
    text_ = re.sub(r'[⁄/]', '/', text_)
    text_ = re.sub(r'\s+', ' ', text_).strip()
    return text_


def standardize_date_format(date_value):
    if pd.isna(date_value):
        return None

    date_str = str(date_value).strip()

    if date_str in ['None', 'nan', '', 'NaT']:
        return None

    try:
        match = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', date_str)
        if match:
            year, part1, part2 = match.groups()
            if int(part1) > 12 and int(part2) <= 12:
                try:
                    parsed_date = pd.to_datetime(
                        f"{year}-{part2.zfill(2)}-{part1.zfill(2)}",
                        format='%Y-%m-%d', errors='raise'
                    )
                    return parsed_date.strftime('%Y-%m-%d')
                except Exception:
                    return date_str
            else:
                return date_str[:10]

        if '/' in date_str:
            parts = date_str.split('/')
            if len(parts) == 3:
                part1, part2, year = parts

                year = year.strip()
                if len(year) == 2:
                    year_int = int(year)
                    year = f"20{year}" if year_int <= 30 else f"19{year}"
                elif len(year) == 3:
                    year = f"20{year[1:]}" if year.startswith('2') else f"20{year}"
                elif len(year) == 1:
                    year = f"202{year}"

                year = year.zfill(4)
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                month, day = part1, part2

                try:
                    parsed_date = pd.to_datetime(
                        f"{year}-{month}-{day}", format='%Y-%m-%d', errors='raise'
                    )
                    return parsed_date.strftime('%Y-%m-%d')
                except Exception:
                    try:
                        parsed_date = pd.to_datetime(
                            f"{year}-{day}-{month}", format='%Y-%m-%d', errors='raise'
                        )
                        return parsed_date.strftime('%Y-%m-%d')
                    except Exception:
                        return date_str

        for fmt in ['%m/%d/%Y', '%m/%d/%y', '%d/%m/%Y', '%d/%m/%y', '%Y-%m-%d', '%d-%m-%Y']:
            try:
                parsed_date = pd.to_datetime(date_str, format=fmt, errors='raise')
                return parsed_date.strftime('%Y-%m-%d')
            except Exception:
                continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parsed_date = pd.to_datetime(date_str, dayfirst=False, errors='coerce')
            if not pd.isna(parsed_date) and parsed_date.year > 1900:
                return parsed_date.strftime('%Y-%m-%d')

    except Exception as e:
        print(f"⚠ Lỗi parse ngày '{date_str}': {e} - GIỮ NGUYÊN")
        return date_str

    return date_str


def _eval_simple(expr: str):
    allowed_ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
    }

    def _clean_num(s):
        s = s.strip()
        if s.count(',') and s.count('.') == 0:
            s = s.replace(',', '')
        elif s.count('.') > 1 and s.count(',') == 0:
            s = s.replace('.', '')
        if ',' in s and '.' not in s and len(s.split(',')[-1]) <= 2:
            s = s.replace(',', '.')
        return s

    try:
        tree = ast.parse(expr, mode='eval')
    except Exception:
        return None

    def _convert(node):
        if isinstance(node, ast.Expression):
            return _convert(node.body)
        if isinstance(node, ast.Num):
            return Decimal(str(node.n))
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return Decimal(str(node.value))
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_ops:
            left = _convert(node.left)
            right = _convert(node.right)
            if left is None or right is None:
                return None
            try:
                return Decimal(str(allowed_ops[type(node.op)](left, right)))
            except Exception:
                return None
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            v = _convert(node.operand)
            return -v if v is not None else None
        if isinstance(node, ast.Call):
            return None
        if isinstance(node, ast.Name):
            return None
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            cleaned = _clean_num(node.value)
            try:
                return Decimal(cleaned)
            except Exception:
                return None
        return None

    cleaned_expr = expr.strip()
    cleaned_expr = re.sub(r'\s+', '', cleaned_expr)
    cleaned_expr = re.sub(r'[\d][\d.,]*', lambda m: _clean_num(m.group(0)), cleaned_expr)

    try:
        tree = ast.parse(cleaned_expr, mode='eval')
    except Exception:
        return None
    return _convert(tree)


def clean_currency_column(series):
    series_str = series.astype(str).str.replace(r'[^\d,\.]', '', regex=True).str.strip()

    mask_vn_format = series_str.str.contains(r'\.') & series_str.str.contains(r',')
    series_str[mask_vn_format] = (
        series_str[mask_vn_format]
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    mask_comma_only = (~series_str.str.contains('\.')) & series_str.str.contains(',')
    series_str[mask_comma_only] = series_str[mask_comma_only].str.replace(',', '.', regex=False)

    mask_multi_dot = series_str.str.count('\.') > 1
    if mask_multi_dot.any():
        def fix_multi_dot(x: str):
            parts = x.split('.')
            last = parts[-1]
            if len(last) <= 2 and all(p.isdigit() for p in parts[:-1]):
                head = ''.join(parts[:-1])
                return head + '.' + last
            else:
                return x.replace('.', '')
        series_str[mask_multi_dot] = series_str[mask_multi_dot].apply(fix_multi_dot)

    mask_single_dot = series_str.str.count('\.') == 1
    series_str[mask_single_dot] = series_str[mask_single_dot].apply(
        lambda x: x if len(x.split('.')[1]) <= 2 else x.replace('.', '')
    )

    numeric_values = pd.to_numeric(series_str, errors='coerce')
    numeric_values = numeric_values.apply(lambda x: x * 1000 if pd.notna(x) and x < 1000 else x)

    originally_na = series.isna()
    numeric_values[originally_na] = pd.NA

    return numeric_values.round(0).astype('Int64')


def clean_whitespace_and_newlines_capdon(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = (
                df[col]
                .str.replace(r'[\n\r\t]+', '', regex=True)
                .str.replace(r'\s{2,}', ' ', regex=True)
                .str.strip()
            )
    return df


In [ ]:
def get_all_data(excel_source):
    """excel_source: đường dẫn file HOẶC pd.ExcelFile / BytesIO chứa toàn bộ workbook cấp đơn."""
    bhsk_columns = ['Ngày update', 'STT', 'Tên Người được BH', 'Ngày Sinh', 'Giới tính', 'Email',
                    'Số hộ chiếu', 'CMND', 'Địa chỉ', 'Tên', 'Quan hệ', 'Ngày sinh người mua Bảo hiểm',
                    'Số CMND_CCCD NMBH', 'Số điện thoại NMBH', 'Địa chỉ NMBH', 'Email NMBH',
                    'Chương trình bảo hiểm', 'Ngoại trú', 'Nha khoa', 'Thai sản', 'Topup',
                    'Phí bảo hiểm', 'Giảm phí_refund', 'Giảm phí_deduct', 'Tổng giảm phí',
                    'Số tiền thanh toán', 'Ngày thanh toán', 'Ngày bắt đầu', 'Ngày kết thúc',
                    'Số Giấy Chứng Nhận', 'Số hợp đồng', 'Thông tin xuất hoá đơn',
                    'Phí điều chỉnh ( Nếu có)', 'Giảm phí ( Nếu có)', 'Nguyên nhân (Nếu có)',
                    'Lead ID (nếu có)', 'Phone trên lead', 'Code sale', 'Phone Khách hàng',
                    'Tên liên hệ', 'Người giới thiệu', 'Hình thức thanh toán', 'Note',
                    'Đối tác nhà bảo hiểm', 'Sản phẩm', 'Channel', 'Mã hợp đồng Affina',
                    'Đã gửi mail cho khách', 'Nộp claim', 'Ủy quyền bồi thường (Nếu có)',
                    'Cột_51', 'Cột_52', 'Cột_53', 'Cột_54', 'Cột_55', 'Cột_56', 'Cột_57',
                    'Cột_58', 'Cột_59', 'Cột_60']

    df_BHSK = pd.read_excel(excel_source, sheet_name='Sức khỏe', header=None, skiprows=5)
    df_BHSK.columns = bhsk_columns[:len(df_BHSK.columns)]
    df_BHSK['Loại bảo hiểm'] = 'BHSK'

    date_cols_bhsk = ['Ngày bắt đầu', 'Ngày kết thúc', 'Ngày Sinh', 'Ngày sinh người mua Bảo hiểm', 'Ngày thanh toán']
    for col in date_cols_bhsk:
        if col in df_BHSK.columns:
            df_BHSK[col] = df_BHSK[col].apply(standardize_date_format)

    df_BHSK = df_BHSK.rename(columns={
        'Tên': 'Tên NMBH',
        'Giới tính': 'Giới tính NNBH',
        'Ngày sinh người mua Bảo hiểm': 'Ngày sinh NMBH',
        'Ngày Sinh': 'Ngày Sinh NNBH',
        'Chương trình bảo hiểm': 'Gói bảo hiểm',
        'CMND': 'CCCD'
    })

    cols_bhsk = ['STT', 'Ngày thanh toán', 'Code sale', 'Sản phẩm', 'Quan hệ', 'Chương trình bảo hiểm',
                 'Ngoại trú', 'Nha khoa', 'Thai sản', 'Topup', 'Đối tác nhà bảo hiểm', 'Số CMND_CCCD NMBH',
                 'Số tiền thanh toán', 'Channel', 'Tên Người được BH', 'Số hộ chiếu', 'CCCD',
                 'Ngày Sinh NNBH', 'Giới tính NNBH', 'Địa chỉ NMBH', 'Phone Khách hàng', 'Email',
                 'Ngày bắt đầu', 'Ngày kết thúc', 'Loại bảo hiểm', 'Số hợp đồng', 'Tên NMBH',
                 'Ngày sinh NMBH', 'Gói bảo hiểm', 'Địa chỉ']
    df_BHSK = df_BHSK[[col for col in cols_bhsk if col in df_BHSK.columns]]

    if 'Số tiền thanh toán' in df_BHSK.columns:
        df_BHSK['Số tiền thanh toán'] = clean_currency_column(df_BHSK['Số tiền thanh toán'])

    bhxm_columns = ['Ngày update', 'STT', 'BIỂN SỐ XE', 'SỐ KHUNG', 'SỐ MÁY', 'TÊN KHÁCH HÀNG',
                    'PHÍ BẢO HIỂM TNDS BẮT BUỘC', 'PHÍ BẢO HIỂM TAI NẠN NNTX', 'SỐ NĂM',
                    'TỔNG PHÍ BẢO HIỂM', 'NGÀY CẤP ĐƠN', 'NGÀY BẮT ĐẦU', 'NGÀY KẾT THÚC',
                    'LOẠI XE', 'NHÃN HIỆU XE', 'SỐ ĐIẸN THOẠI', 'Email', 'Chương trình',
                    'Code sale', 'Hình thức thanh toán', 'Đối tác nhà bảo hiểm', 'Sản phẩm',
                    'Channel', 'Số hợp đồng', 'Note']

    df_BHXM = pd.read_excel(excel_source, sheet_name='Thông tin cấp Bảo hiểm xe máy', header=None, skiprows=2)
    df_BHXM.columns = bhxm_columns[:len(df_BHXM.columns)]

    df_BHXM = df_BHXM.dropna(subset=['Ngày update'])

    date_cols_bhxm = ['Ngày update', 'NGÀY CẤP ĐƠN', 'NGÀY BẮT ĐẦU', 'NGÀY KẾT THÚC']
    for col in date_cols_bhxm:
        if col in df_BHXM.columns:
            df_BHXM[col] = df_BHXM[col].apply(standardize_date_format)

    cols_to_select = ['Ngày update', 'Code sale', 'Chương trình', 'Đối tác nhà bảo hiểm',
                       'TỔNG PHÍ BẢO HIỂM', 'Channel', 'TÊN KHÁCH HÀNG', 'NGÀY BẮT ĐẦU',
                       'NGÀY KẾT THÚC', 'Số hợp đồng', 'BIỂN SỐ XE', 'SỐ ĐIẸN THOẠI', 'Email']

    df_BHXM_done = df_BHXM[[col for col in cols_to_select if col in df_BHXM.columns]].copy()

    if not df_BHXM_done.empty:
        df_BHXM_done = df_BHXM_done.rename(columns={
            'Ngày update': 'Ngày thanh toán',
            'Chương trình': 'Sản phẩm',
            'TỔNG PHÍ BẢO HIỂM': 'Số tiền thanh toán',
            'TÊN KHÁCH HÀNG': 'Tên Người được BH',
            'NGÀY BẮT ĐẦU': 'Ngày bắt đầu',
            'NGÀY KẾT THÚC': 'Ngày kết thúc',
            'BIỂN SỐ XE': 'Biển số xe',
            'SỐ ĐIẸN THOẠI': 'Phone Khách hàng'
        })

        if 'Tên Người được BH' in df_BHXM_done.columns:
            df_BHXM_done['Tên NMBH'] = df_BHXM_done['Tên Người được BH']

        df_BHXM_done['Loại bảo hiểm'] = 'BHXM'
        df_BHXM_done['Số tiền thanh toán'] = clean_currency_column(df_BHXM_done['Số tiền thanh toán'])

    df_BHYT = pd.read_excel(excel_source, sheet_name='BHYTBHXH', skiprows=1, dtype={"Số tiền thanh toán": str})

    col_count = {}
    new_columns = []
    for col in df_BHYT.columns:
        base_col = col.split('.')[0]

        if base_col not in col_count:
            col_count[base_col] = 0
            new_columns.append(base_col)
        else:
            col_count[base_col] += 1
            if base_col == 'Ngày sinh':
                if col_count[base_col] == 1:
                    new_columns.append('Ngày sinh NMBH')
            elif base_col == 'CCCD':
                if col_count[base_col] == 1:
                    new_columns.append('CCCD NMBH')
            elif base_col == 'Địa chỉ':
                if col_count[base_col] == 1:
                    new_columns.append('Địa chỉ NMBH')
            elif base_col == 'SĐT':
                if col_count[base_col] == 1:
                    new_columns.append('SĐT NMBH')
            elif base_col == 'Email':
                if col_count[base_col] == 1:
                    new_columns.append('Email NMBH')
            elif base_col == 'Sản phẩm':
                if col_count[base_col] == 1:
                    new_columns.append('Sản phẩm 2')
            else:
                new_columns.append(f"{base_col}_{col_count[base_col]}")

    df_BHYT.columns = new_columns
    df_BHYT['Loại bảo hiểm'] = 'BHYT/BHXH'

    if 'Note' in df_BHYT.columns:
        df_BHYT['Note'] = df_BHYT['Note'].astype(str).replace({'nan': pd.NA, 'None': pd.NA})

    date_cols_bhyt = ['Ngày thanh toán', 'Ngày bắt đầu', 'Ngày kết thúc', 'Ngày sinh',
                       'Ngày sinh NMBH', 'Ngày update', 'Ngày duyệt', 'Ngày hoàn phí']
    for col in date_cols_bhyt:
        if col in df_BHYT.columns:
            df_BHYT[col] = df_BHYT[col].apply(standardize_date_format)

    rename_dict = {
        'Họ tên NĐBH': 'Tên Người được BH',
        'Họ tên BMBH': 'Tên NMBH',
        'Ngày sinh': 'Ngày Sinh NNBH',
        'Code sales': 'Code sale',
        'Đối tác NBH': 'Đối tác nhà bảo hiểm',
        'Mã tờ khai': 'Số hợp đồng',
        'Phí Bảo hiểm': 'Số tiền thanh toán',
        'Ngày duyệt': 'Ngày thanh toán',
    }

    df_BHYT = df_BHYT.rename(columns={k: v for k, v in rename_dict.items() if k in df_BHYT.columns})

    cols_bhyt = [
        'STT', 'Ngày thanh toán', 'Code sale', 'Sản phẩm', 'Đối tác nhà bảo hiểm',
        'Số tiền thanh toán', 'Channel', 'Tên Người được BH', 'Ngày Sinh NNBH',
        'Giới tính', 'CCCD', 'Địa chỉ', 'SĐT', 'Email',
        'Phone Khách hàng', 'Ngày bắt đầu', 'Ngày kết thúc', 'Loại bảo hiểm',
        'Số hợp đồng', 'Tên NMBH', 'Ngày sinh NMBH', 'CCCD NMBH', 'Địa chỉ NMBH',
        'SĐT NMBH', 'Email NMBH', 'Mã BHYT', 'Mã BHXH',
        'Trạng thái', 'Ngày thanh toán (real)', 'Ngày hoàn phí', 'Phương án thù lao',
        'Phí Bảo hiểm', 'Thời hạn BH', 'Số GCN', 'Tên liên hệ', 'Hình thức thanh toán'
    ]
    df_BHYT_done = df_BHYT[[col for col in cols_bhyt if col in df_BHYT.columns]]

    if 'Số tiền thanh toán' in df_BHYT_done.columns:
        df_BHYT_done['Số tiền thanh toán'] = clean_currency_column(df_BHYT_done['Số tiền thanh toán'])

    if 'Phí Bảo hiểm' in df_BHYT_done.columns:
        df_BHYT_done['Phí Bảo hiểm'] = clean_currency_column(df_BHYT_done['Phí Bảo hiểm'])

    df_BHOTO = pd.read_excel(excel_source, sheet_name='Bao hiem oto', dtype={"Số tiền": str})

    if len(df_BHOTO.columns) > 30:
        col_ae_name = df_BHOTO.columns[30]
        df_BHOTO = df_BHOTO.rename(columns={col_ae_name: 'Note'})
    else:
        df_BHOTO['Note'] = pd.NA

    df_BHOTO_done = df_BHOTO[['Ngày thanh toán', 'Code sale', 'Chương trình', 'Đối tác nhà bảo hiểm',
                               'Số tiền', 'Channel', 'Tên khách hàng', 'Số GCN', 'Biển số',
                               'Ngày bắt đầu hiệu lực', 'Ngày kết thúc hiệu lực', 'Số điện thoại', 'email', 'Note']].copy()

    df_BHOTO_done = df_BHOTO_done.rename(columns={
        'Chương trình': 'Sản phẩm',
        'Số tiền': 'Số tiền thanh toán',
        'Tên khách hàng': 'Tên Người được BH',
        'Số GCN': 'Số hợp đồng',
        'Biển số': 'Biển số xe',
        'Ngày bắt đầu hiệu lực': 'Ngày bắt đầu',
        'Ngày kết thúc hiệu lực': 'Ngày kết thúc',
        'Số điện thoại': 'Phone Khách hàng',
        'email': 'Email'
    })

    df_BHOTO_done = df_BHOTO_done.loc[:, ~df_BHOTO_done.columns.duplicated()]

    df_BHOTO_done['Tên NMBH'] = df_BHOTO_done['Tên Người được BH']
    df_BHOTO_done['Loại bảo hiểm'] = 'BHOTO'
    df_BHOTO_done['Số tiền thanh toán'] = clean_currency_column(df_BHOTO_done['Số tiền thanh toán'])

    date_cols_bhoto = ['Ngày thanh toán', 'Ngày bắt đầu', 'Ngày kết thúc', 'Ngày sinh',
                        'Ngày sinh NMBH', 'Ngày update', 'Ngày duyệt', 'Ngày hoàn phí']
    for col in date_cols_bhoto:
        if col in df_BHOTO_done.columns:
            df_BHOTO_done[col] = df_BHOTO_done[col].apply(standardize_date_format)

    df_BHDL = pd.read_excel(excel_source, sheet_name='Du lịch', dtype={"Phí bảo hiểm": str})
    df_BHDL = df_BHDL.rename(columns={'Họ Và Tên': 'Tên Người được BH'})
    df_BHDL_done = df_BHDL[['Ngày thanh toán', 'Tên sale', 'Sản phẩm', 'Đối tác nhà BH',
                             'Phí bảo hiểm', 'Channel', 'Tên Người được BH', 'Số hợp đồng']].copy()
    df_BHDL_done['Loại sản phẩm'] = 'BHDL'
    df_BHDL_done.columns = ['Ngày thanh toán', 'Code sale', 'Sản phẩm', 'Đối tác nhà bảo hiểm',
                             'Số tiền thanh toán', 'Channel', 'Tên Người được BH', 'Số hợp đồng', 'Loại bảo hiểm']
    df_BHDL_done['Số tiền thanh toán'] = clean_currency_column(df_BHDL_done['Số tiền thanh toán'])

    for col in date_cols_bhoto:
        if col in df_BHDL_done.columns:
            df_BHDL_done[col] = df_BHDL_done[col].apply(standardize_date_format)

    df_TNSP = pd.read_excel(excel_source, sheet_name='Trách nhiệm sản phẩm', dtype={"Phí Bảo hiểm": str})
    df_TNSP = df_TNSP.rename(columns={'Nhà Bảo hiểm': 'Đối tác nhà bảo hiểm', 'Phí Bảo hiểm': 'Số tiền thanh toán'})
    df_TNSP['Loại bảo hiểm'] = 'TNDS'
    df_TNSP_done = df_TNSP[['Ngày thanh toán', 'Code sale', 'Sản phẩm', 'Đối tác nhà bảo hiểm',
                             'Số tiền thanh toán', 'Channel', 'Loại bảo hiểm']]
    df_TNSP_done['Số tiền thanh toán'] = clean_currency_column(df_TNSP_done['Số tiền thanh toán'])

    df_BHRR = pd.read_excel(excel_source, sheet_name='Bảo hiểm rủi ro', dtype={"Số tiền thanh toán": str})

    date_cols_bhrr = ['Ngày thanh toán', 'Ngày bắt đầu', 'Ngày kết thúc']
    for col in date_cols_bhrr:
        if col in df_BHRR.columns:
            df_BHRR[col] = df_BHRR[col].apply(standardize_date_format)

    df_BHRR = df_BHRR.rename(columns={'Tên khách hàng': 'Tên Người được BH', 'Mã hợp đồng': 'Số hợp đồng'})
    df_BHRR['Loại bảo hiểm'] = 'BHRR'

    df_BHRR_done = df_BHRR[['Ngày thanh toán', 'Code sale', 'Sản phẩm', 'Đối tác nhà bảo hiểm',
                             'Số tiền thanh toán', 'Channel', 'Tên Người được BH', 'Số hợp đồng',
                             'Ngày bắt đầu', 'Ngày kết thúc', 'Loại bảo hiểm']]
    df_BHRR_done['Số tiền thanh toán'] = clean_currency_column(df_BHRR_done['Số tiền thanh toán'])

    df_union = pd.concat([df_BHSK, df_BHOTO_done, df_BHXM_done, df_BHDL_done,
                           df_TNSP_done, df_BHRR_done, df_BHYT_done], axis=0, ignore_index=True)

    return df_union


In [ ]:
if RUN_UPDATE_CAPDON:
    capdon_bytes = download_google_sheet_as_excel_bytes(CAPDON_SHEET_URL)
    capdon_excel = pd.ExcelFile(io.BytesIO(capdon_bytes))

    df_union = get_all_data(capdon_excel)

    rows_before = len(df_union)
    print(f"📊 Số dòng TRƯỚC xử lý: {rows_before}")

    df_union["Số hợp đồng"] = df_union["Số hợp đồng"].apply(clean_contract)

    mask = (
        (df_union["Loại bảo hiểm"] == "BHYT/BHXH")
        & (df_union["Số hợp đồng"].notna())
    )

    def add_dot_zero(x):
        x = str(x).strip()
        if x == "" or x == "-":
            return x
        if x.endswith(".0"):
            return x
        return x + ".0"

    df_union.loc[mask, "Số hợp đồng"] = df_union.loc[mask, "Số hợp đồng"].apply(add_dot_zero)

    df_union['Ngày thanh toán'] = df_union['Ngày thanh toán'].apply(standardize_date_format)
    df_union = clean_whitespace_and_newlines_capdon(df_union)

    for col in ['Ngày bắt đầu', 'Ngày kết thúc', 'Ngày Sinh NNBH', 'Ngày sinh NMBH']:
        if col in df_union.columns:
            df_union[col] = df_union[col].apply(standardize_date_format)

    rows_after = len(df_union)
    print(f"📊 Số dòng SAU xử lý: {rows_after}")

    if rows_before != rows_after:
        print(f"⚠️ CẢNH BÁO: Mất {rows_before - rows_after} dòng!")
    else:
        print(f"✅ Giữ nguyên toàn bộ {rows_after} dòng")

    df_union.to_sql('union_all_data_cap_don', index=False, if_exists='replace', con=engine)
    print(f"✔ Đã cập nhật {len(df_union)} dòng vào bảng union_all_data_cap_don (Supabase)")
else:
    print("⏭ Bỏ qua Bước 2 (RUN_UPDATE_CAPDON = False)")


## Bước 3 — Chạy 3 câu query trên Supabase và xuất ra 1 file Excel (3 sheet)

3 file `.sql` được tải trực tiếp từ link (Google Drive public / GitHub) mỗi lần chạy, nên bạn chỉ cần sửa nội dung trên Drive/GitHub, không cần sửa lại notebook.

In [ ]:
import re
import requests


def _normalize_sql_url(url: str) -> str:
    """Chuyển link Google Drive / GitHub (dạng xem trên web) thành link tải nội dung thô."""
    url = url.strip()

    # Google Drive: .../file/d/<ID>/view... hoặc .../open?id=<ID> hoặc .../uc?id=<ID>
    m = re.search(r"drive\.google\.com.*?/d/([a-zA-Z0-9_-]+)", url)
    if not m:
        m = re.search(r"drive\.google\.com.*?[?&]id=([a-zA-Z0-9_-]+)", url)
    if m:
        file_id = m.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"

    # GitHub: .../blob/<branch>/<path> -> raw.githubusercontent.com/.../<branch>/<path>
    m = re.search(r"github\.com/([^/]+)/([^/]+)/blob/(.+)", url)
    if m:
        owner, repo, rest = m.groups()
        return f"https://raw.githubusercontent.com/{owner}/{repo}/{rest}"

    # GitHub Gist trang xem -> thêm /raw nếu chưa có (tuỳ chọn, chỉ áp dụng khi rõ ràng là gist)
    if "gist.github.com" in url and not url.rstrip("/").endswith("raw"):
        return url.rstrip("/") + "/raw"

    # Mặc định: coi như đã là link tải trực tiếp (raw.githubusercontent.com, Supabase storage, ...)
    return url


def download_sql_text(url: str, label: str = "") -> str:
    direct_url = _normalize_sql_url(url)
    resp = requests.get(direct_url, timeout=60)
    resp.raise_for_status()
    resp.encoding = resp.encoding or "utf-8"
    content = resp.text

    # Cảnh báo nếu link Google Drive trả về trang xác nhận virus-scan thay vì file thật
    if "<html" in content[:500].lower():
        raise ValueError(
            f"Không tải được nội dung SQL thật cho '{label}' từ URL: {url}\n"
            f"(nhận được HTML thay vì SQL — kiểm tra lại quyền chia sẻ 'Anyone with the link' "
            f"hoặc dùng link GitHub raw thay thế)"
        )
    return content


SQL_QUERIES = {
    "Contest_GDKD": download_sql_text(SQL_GDKD_URL, "Contest_GDKD"),
    "Contest_CVKD_latest": download_sql_text(SQL_CVKD_LATEST_URL, "Contest_CVKD_latest"),
    "Contest_TPKD_latest": download_sql_text(SQL_TPKD_LATEST_URL, "Contest_TPKD_latest"),
}

for _name, _sql in SQL_QUERIES.items():
    print(f"✔ Đã tải {_name}: {len(_sql)} ký tự")


In [ ]:
if RUN_QUERIES:
    results = {}

    with engine.connect() as conn:
        for sheet_name, sql_text_content in SQL_QUERIES.items():
            print(f"▶ Đang chạy: {sheet_name} ...")
            df_result = pd.read_sql(text(sql_text_content), conn)
            results[sheet_name] = df_result
            print(f"   → {len(df_result)} dòng, {len(df_result.columns)} cột")

    with pd.ExcelWriter(OUTPUT_EXCEL_PATH, engine="openpyxl") as writer:
        for sheet_name, df_result in results.items():
            safe_name = sheet_name[:31]  # Excel giới hạn tên sheet 31 ký tự
            df_result.to_excel(writer, sheet_name=safe_name, index=False)

    print(f"\n✅ Đã xuất kết quả ra file: {OUTPUT_EXCEL_PATH}")

    # Tải file về máy (chạy trên Colab)
    try:
        from google.colab import files as _colab_files
        _colab_files.download(OUTPUT_EXCEL_PATH)
    except ImportError:
        pass
else:
    print("⏭ Bỏ qua Bước 3 (RUN_QUERIES = False)")
